# Dataset inspection

Read what the model is actually being trained on: the mix, the refusals, the
noisy contexts, and the exact prompt the trainer builds.

Run from the repository root.

In [ ]:
import sys, collections
sys.path.insert(0, "..")

from src.dataset.io import read_jsonl
from src.prompt import build_messages, SYSTEM_PROMPT

train = read_jsonl("../data/train/train.jsonl")
test = read_jsonl("../data/test/test.jsonl")
print(len(train), "train /", len(test), "test")
collections.Counter(e.answerable for e in train)

## Category and project mix

In [ ]:
import collections
print(collections.Counter(e.category for e in train).most_common())
print(collections.Counter(e.project for e in train))
print("train projects:", {e.project for e in train})
print("test projects :", {e.project for e in test})  # disjoint by construction

## A grounded answer, and the prompt the trainer builds from it

In [ ]:
ex = next(e for e in train if e.answerable == "full" and len(e.context) >= 3)
print(ex.question)
print()
print(ex.answer)

In [ ]:
messages = build_messages(ex.question, [c.model_dump() for c in ex.context], ex.answer)
print(messages[1]["content"][:1500])

## A refusal, and what makes it correct

In [ ]:
neg = next(e for e in train if e.answerable == "none")
print(neg.question)
print()
print(neg.answer)
print()
print("sources in context:", [f"{c.source}#{c.heading}" for c in neg.context])

## Noisy contexts: how many sections are irrelevant

In [ ]:
noisy = [e for e in train if any(c.relevant is False for c in e.context)]
print(len(noisy), "examples carry distractors")
e = noisy[0]
for c in e.context:
    print(("  relevant  " if c.relevant is not False else "  DISTRACTOR"), f"[{c.source} — {c.heading}]")

## Score the reference answers with the evaluators (sanity check)

In [ ]:
from src.evaluation.metrics import evaluate_example, aggregate
rows = [evaluate_example(e, e.answer) for e in test]
{k: v for k, v in aggregate(rows).items() if not k.startswith("length")}